# 14 Spark UI and Query Profile

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Know what each tab of the Spark UI shows, what to look at first when a job is slow, and how to get the same insight on Databricks serverless from the <b>query profile</b> and <b>query history</b>. Then practise by reading the profiles of queries that have deliberately built-in problems.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Every performance discussion ends with "show me the Spark UI". Interviewers ask "a job is slow, how do you debug it?", and the strong answer is a concrete walk through jobs, stages, task metrics, shuffle and spill. The certification's "Troubleshooting, Monitoring and Optimization" domain (about 10%) tests exactly this.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The Spark UI (classic compute)</b><br>
| Tab | Shows | Look for |<br>|---|---|---|<br>| <b>Jobs</b> | One row per action, with stages and duration | The slowest job, failed jobs |<br>| <b>Stages</b> | Per stage: tasks, input, shuffle read/write, spill, and the task metric distribution (min / median / max) | <b>Max ≫ median</b> task time (skew), large shuffle, spill |<br>| <b>SQL / DataFrame</b> | The query's plan as a graph with per-operator metrics (rows, time, sizes), and the <b>final</b> AQE plan | Which operator takes the time, rows exploding after a join |<br>| <b>Storage</b> | Cached DataFrames and how much is in memory or on disk | Unexpected or stale caches |<br>| <b>Environment</b> | Spark configs and versions | The settings that were actually used |<br>| <b>Executors</b> | Per executor: tasks, GC time, memory, shuffle, failures | High GC time, lost executors, idle executors |
</div>
<br>
<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The query profile (serverless and SQL warehouses)</b><br>
On serverless you don't get the Spark UI. Instead:
<ul><li>Under a cell's output, open <b>See performance → the query</b> to view its <b>query profile</b></li><li>Or open <b>Query History</b> in the sidebar, filter by your notebook or user, and click a query</li></ul>
The profile shows the query as a graph of operators (often <code>Photon...</code> operators) with <b>time spent</b>, <b>rows produced</b>, <b>bytes</b>, <b>memory</b>, <b>spill</b>, and, for exchanges, shuffle sizes. It is the final, adapted plan, so AQE's changes are visible.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
"The nightly job got slow." The on-call engineer opens the run, finds the slowest stage, and sees task durations min 4 s / median 6 s / max 38 min. That one task read 45 GB of shuffle data while the median read 300 MB: a skewed join key. The SQL tab shows the join and its input sizes, and the fix (AQE skew handling plus a salted key) is clear within 15 minutes, without guessing.
</div>

## The debugging checklist

```
1. Which JOB / query is slow?                     Jobs tab · Query History (sort by duration)
2. Which STAGE / operator dominates?               Stages tab · query profile "time spent"
3. Are TASKS balanced?                             min / median / max duration and input   -> skew?
4. How much is SHUFFLED? Is there SPILL?           shuffle read/write, spill (memory/disk)  -> too much data, too few partitions?
5. Too many / too few TASKS?                       task count vs cores                       -> partition sizing
6. Is the PLAN what you expect?                    SQL tab / profile: join strategy, filters pushed, rows per operator
7. Executors healthy?                              GC time, failures, idle executors
```

## Setup: specimen queries

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines five small "specimen" queries, each with a typical behaviour to recognize in a profile, and runs them one by one.<br><br>
<b>Why it matters</b><br>Reading profiles is a skill learned by looking at them. Run each cell, open its query profile (serverless) or the Spark UI SQL and Stages tabs (classic), and write what you see in the observation log at the end.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Each cell prints a label and a duration. The interesting part is in the profile.
</div>

In [0]:
import time
from pyspark.sql import functions as F

def run(label, df):
    start = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:<40} {time.time() - start:6.2f}s  <- open this query's profile")

events = (
    spark.range(0, 4_000_000, 1, numPartitions=8)
    .select(
        F.col("id").alias("event_id"),
        (F.col("id") % 200_000).alias("user_id"),
        (F.col("id") % 40).alias("page"),
        F.sha2(F.col("id").cast("string"), 256).alias("payload"),
    )
)

## Specimen 1: a healthy aggregation

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
A filter and a <code>groupBy</code> with partial aggregation.<br><br>
<b>What to look for</b><br>Two stages around one exchange. Rows drop sharply after the partial aggregate. Task times are even, and there's no spill.
</div>

In [0]:
run("1. healthy aggregation", events.filter("page < 20").groupBy("page").agg(F.count("*"), F.countDistinct("user_id")))

## Specimen 2: a heavy shuffle

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deduplicates while carrying a wide <code>payload</code> column through the shuffle.<br><br>
<b>What to look for</b><br>Large shuffle write and read on the exchange. Compare its bytes with specimen 1. The fix would be to select fewer columns before the shuffle.
</div>

In [0]:
run("2. heavy shuffle (wide rows)", events.dropDuplicates(["user_id"]))

## Specimen 3: a skewed join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins events to a lookup where 85% of the rows share one key.<br><br>
<b>What to look for</b><br>In the stage or operator details, the <b>max</b> task duration and input are far above the <b>median</b>. On recent runtimes AQE may split the skewed partition: look for a note about skew in the join or shuffle-read node.
</div>

In [0]:
skewed = events.withColumn("seg", F.when(F.col("event_id") % 100 < 85, F.lit(0)).otherwise(F.col("event_id") % 500))
segments = spark.range(500).withColumnRenamed("id", "seg").withColumn("name", F.concat(F.lit("segment_"), F.col("seg")))
run("3. skewed join", skewed.join(segments.hint("merge"), "seg"))

## Specimen 4: a global sort

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sorts all rows by a high-cardinality column.<br><br>
<b>What to look for</b><br>A sampling job before the main job (range boundaries), an exchange with <b>range</b> partitioning, and a sort operator with high time and memory. With large data and few partitions, this is where <b>spill</b> appears.
</div>

In [0]:
run("4. global sort", events.orderBy("payload"))

## Specimen 5: a broadcast join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins events to a small dimension, which Spark broadcasts.<br><br>
<b>What to look for</b><br>A broadcast exchange of a few KB, and <b>no</b> shuffle of the events side. Compare with specimen 3, where both sides were shuffled.
</div>

In [0]:
pages = spark.range(40).withColumnRenamed("id", "page").withColumn("section", F.col("page") % 4)
run("5. broadcast join", events.join(pages, "page").groupBy("section").count())

## Where is the UI on this compute?

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tries to print the Spark UI address (classic compute) and otherwise explains where to look.<br><br>
<b>Why it matters</b><br>On classic clusters the UI is linked from the cluster page and the notebook's compute menu. On serverless, the query profile and Query History replace it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
On serverless or Spark Connect: a message pointing to the query profile. On classic compute: a URL.
</div>

In [0]:
try:
    print("Spark UI:", spark.sparkContext.uiWebUrl)
except Exception:
    print("No Spark UI on this compute (serverless / Spark Connect).")
    print("Use: cell output -> 'See performance' -> query profile, or the Query History page in the sidebar.")

## Query history from SQL (system tables)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries the <code>system.query.history</code> system table for your most recent statements.<br><br>
<b>Why it matters</b><br>System tables let you analyse query durations, bytes read and spill across many runs with SQL, which is the basis of monitoring dashboards (covered in <code>04_databricks/08_monitoring_basics</code>).
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Access to system tables depends on your workspace. If the query fails, use the Query History page in the UI instead.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your latest statements with duration and bytes read, or a message that the table isn't accessible.
</div>

In [0]:
try:
    display(spark.sql("""
        SELECT start_time, statement_type, total_duration_ms, read_bytes, spilled_local_bytes,
               left(statement_text, 80) AS statement
        FROM system.query.history
        WHERE executed_by = current_user()
        ORDER BY start_time DESC
        LIMIT 10
    """))
except Exception as e:
    print("system.query.history not accessible here:", type(e).__name__)

## My observation log

Fill this in from the profiles of the five specimens.

| Specimen | Duration | Stages / exchanges | Shuffle bytes | Max vs median task (or note) | Spill? | Join strategy |
|---|---|---|---|---|---|---|
| 1. healthy aggregation | | | | | | n/a |
| 2. heavy shuffle | | | | | | n/a |
| 3. skewed join | | | | | | |
| 4. global sort | | | | | | n/a |
| 5. broadcast join | | | | | | |

## Under the hood

```
Spark UI data flow:  executors send task metrics ─▶ driver's listener bus ─▶ UI / event log
                     (completed applications: Spark History Server reads the event log)

Query profile:       Databricks collects per-operator metrics for each query
                     and shows the final (AQE-adapted) plan with time, rows, bytes, spill
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> only actions create entries in the UI or the query history.

**Key metrics glossary**
- **Duration** vs **executor run time**: wall clock vs CPU work. A big gap means waiting (scheduling, fetching shuffle data)
- **Shuffle read/write**: bytes moved by exchanges
- **Spill (memory/disk)**: data that didn't fit in execution memory
- **GC time**: JVM garbage collection. Above about 10% of task time is a warning sign
- **Input / output**: bytes read from and written to storage

In [0]:
events.filter("page < 20").groupBy("page").agg(F.count("*")).explain("formatted")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: looking at the Jobs tab only</b><br>
Jobs show totals. The cause is in the Stages tab (task distribution) or the SQL tab and profile (operators).<br><br>
<b>⛔ Problem: optimizing the wrong thing</b><br>Start from the slowest stage or operator. Don't tune a step that takes 2% of the time.<br><br>
**⛔ Problem: plan in <code>explain()</code> differs from the profile**<br><code>explain()</code> shows the initial plan. The profile shows the final plan after AQE.<br><br>
<b>⛔ Problem: can't find the Spark UI on serverless</b><br>Use "See performance" under the cell, or the Query History page.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. A Spark job is slow. How do you debug it?</b><br>
I find the slowest job and stage in the Spark UI or query profile. Then I check the task distribution in that stage: if the max is far above the median, it's skew. Then shuffle read/write and spill: large shuffles mean too much data moved, and spill means partitions are too big. Then task count against cores, for under- or over-partitioning. Then the SQL plan: join strategy, pushed filters, and rows per operator. And finally executor health, such as GC time and failures. Then I fix the biggest cause and measure again.<br><br>

<b>🎤 2. What are the main tabs of the Spark UI?</b><br>
Jobs, Stages, SQL/DataFrame, Storage, Environment and Executors.<br><br>

<b>🎤 3. How do you spot data skew in the Spark UI?</b><br>
In the Stages tab, the task metric summary shows a max duration and input far above the median, typically one or a few tasks running much longer than the rest.<br><br>

<b>🎤 4. What does spill mean, and what do you do about it?</b><br>
Data that didn't fit in execution memory was written to disk during a sort, aggregation or join. I increase the number of shuffle partitions, reduce the data per row, or fix skew.<br><br>

<b>🎤 5. How do you monitor queries on Databricks serverless?</b><br>
With the query profile of each statement (time, rows, bytes and spill per operator, with the final plan), the Query History page, and system tables such as <code>system.query.history</code> for analysis across runs.<br><br>

<b>🎤 6. Why might the plan in <code>explain()</code> differ from the Spark UI?</b><br>
With AQE, <code>explain()</code> shows the initial plan. The UI's SQL tab and the query profile show the final plan after runtime re-optimization, such as coalesced partitions, converted joins and skew splits.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. In the Spark UI, where do you check whether a few tasks of a stage are much slower than the others?</b><br>
A. Environment tab<br>
B. Storage tab<br>
C. Stages tab, in the task metrics summary (min, median, max)<br>
D. Jobs tab, in the job list only<br>
<details><summary><b>Show answer</b></summary><b>C.</b></details><br><br>

<b>Q2. A data engineer on serverless compute needs to see how much data each operator of a slow query processed. Where should they look?</b><br>
A. The Spark UI Executors tab<br>
B. The query profile of the query (from the cell's performance link or Query History)<br>
C. <code>DESCRIBE HISTORY</code><br>
D. The cluster's driver logs<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Fill in the observation log above for all five specimens</li><li>Make specimen 2 cheaper by selecting only <code>user_id</code> and <code>page</code> before deduplicating. Compare its shuffle bytes in the profile</li><li>For specimen 3, compute the share of rows in the biggest join key, and explain how it relates to the max task time you saw</li><li>Write your own 5-step answer to "a job is slow, how do you debug it?" and say it out loud in under 2 minutes</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 2. Narrow the rows before the shuffle, then compare the exchange bytes in the profile
run("2b. lean shuffle (2 columns)", events.select("user_id", "page").dropDuplicates(["user_id"]))

# 3. Share of the biggest key: the task that gets this key does ~this fraction of the join's work
total = skewed.count()
top = skewed.groupBy("seg").count().orderBy(F.desc("count")).first()
print(f"biggest key seg={top['seg']}: {top['count'] / total:.0%} of all rows")

# 4. Example answer: (1) find the slowest stage/operator, (2) compare max vs median tasks for skew,
#    (3) check shuffle size and spill, (4) check task count vs cores, (5) read the plan: join strategy,
#    pushed filters, rows per operator; fix the biggest cause and measure again.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Spark UI tabs: Jobs, Stages, SQL/DataFrame, Storage, Environment, Executors</li><li>Debug order: slowest stage → task distribution (skew) → shuffle and spill → task count → plan → executors</li><li>Max ≫ median task time = skew. Spill = partitions too big or skewed. High GC = memory pressure</li><li>Serverless: query profile ("See performance") and Query History replace the Spark UI</li><li><code>explain()</code> shows the initial plan. UI and profile show the final AQE plan</li><li><code>system.query.history</code> enables SQL analysis of query metrics</li></ul>
</div>

| Question | Where |
|---|---|
| Which query is slow? | Query History / Jobs tab |
| Which operator? | Query profile / SQL tab |
| Skew? | Stage task summary (max vs median) |
| Shuffle and spill? | Stage metrics / profile exchange and sort nodes |
| Final plan? | SQL tab / query profile |

## Git commit

```
git add 02_spark_internals/14_spark_ui.ipynb
git commit -m "add 02_14 spark ui notebook"
```